# Benchmark: All Models on NASA Ames Multi-Temperature Dataset
## TE-Q-Transformer Research Framework

**Manuscript Title:** *TE-Q-Transformer: A Temperature-Embedded Quantum Framework for Battery State-of-Health Estimation*

This notebook provides the unified evaluation of:
1. **TE-Q-Transformer (Proposed)**: Physics-guided Arrhenius encoding + 4-qubit simulated quantum circuit + Conv1D-Transformer.
2. **10 Contemporary Baseline Models**:
   - Recurrent: **LSTM**, **GRU**
   - Convolutional: **CNN1D**, **TCN** (Causal Dilated)
   - Linear Decomposition: **DLinear**
   - Self-Attention & Transformers: **Transformer**, **PatchTST**, **iTransformer**
   - Quantum-Classical Hybrids: **QLSTM** (Gate-level VQC), **QNN-GRU** (Variational feature map + GRU)

---

### Non-Leakage Evaluation Protocol:
- **Sequence Length:** Exactly 512 time steps ($V$, $I$, $T_C$, $t_{\text{norm}}$).
- **Training Cells:** `B0005`, `B0006`, `B0007`, `B0029`, `B0030`, `B0031`, and first 70% of `B0053` (660 cycles).
- **Testing Cells:** Held-out unseen cells `B0018`, `B0032`, and final 30% of `B0053` (187 cycles).
- **Zero Leakage:** MinMaxScaler fitted **exclusively on training cycles** for channels (0, 1, 3). Temperature (channel 2) is strictly left in unscaled Celsius for physical Arrhenius semantics.


In [ ]:
# ==============================================================================
# 0. CONFIGURABLE REPOSITORY ROOT PATH & ENVIRONMENT SETUP
# ==============================================================================
import os
import sys
from pathlib import Path

# Manual override: Set to Path("your/path") if needed; otherwise auto-discovered.
MANUAL_REPO_ROOT = None
REPO_NAME = "TE-Q-Transformer-A-Temperature-Embedded-Quantum-Framework-for-Battery-State-of-Health-Estimation"

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/kaggle/working") / REPO_NAME,
    Path("/kaggle/working"),
    Path("/content") / REPO_NAME,
    Path("/content"),
]

if MANUAL_REPO_ROOT and Path(MANUAL_REPO_ROOT).exists():
    REPO_ROOT = Path(MANUAL_REPO_ROOT).resolve()
else:
    REPO_ROOT = next(
        (c.resolve() for c in CANDIDATES if (c / "models" / "proposed" / "te_q_transformer.py").exists() or (c / "datasets" / "NASA" / "processed").exists()),
        Path.cwd().resolve()
    )

print(f"[Setup] REPO_ROOT resolved to: {REPO_ROOT}")
DATA_ROOT = REPO_ROOT / "datasets"
MODEL_ROOT = REPO_ROOT / "models"
RESULT_ROOT = REPO_ROOT / "results"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Install PennyLane if running in a fresh cloud runtime (Kaggle/Colab)
try:
    import pennylane as qml
except ImportError:
    print("[Setup] PennyLane not detected. Installing via pip...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pennylane"])
    import pennylane as qml

import random
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Hardware device: {DEVICE}")


## 1. Dataset Loading & Zero-Leakage Preprocessing


In [ ]:
# Resolve NASA Ames dataset directory
def resolve_nasa_data_dir() -> Path:
    candidates = [
        DATA_ROOT / "NASA" / "processed",
        DATA_ROOT / "nasa",
        REPO_ROOT / "Dataset" / "nasa",
        Path("/kaggle/input/nasa-battery-dataset"),
    ]
    for c in candidates:
        if c.is_dir() and (c / "B0005_X.npy").exists():
            return c.resolve()
    for hit in REPO_ROOT.rglob("B0005_X.npy"):
        return hit.parent.resolve()
    raise FileNotFoundError("Could not find NASA Ames dataset files (B0005_X.npy).")

NASA_DIR = resolve_nasa_data_dir()
print(f"[Dataset] Resolved NASA processed directory: {NASA_DIR}")

from src.data.nasa_loader import (
    NASA_TRAIN_CELLS,
    NASA_TEST_CELLS,
    NASA_SPLIT_CELL,
    get_nasa_dataloaders,
)

train_loader, test_loaders, scaler = get_nasa_dataloaders(NASA_DIR, batch_size=8)
print(f"[Dataset] Train batches: {len(train_loader)}")
for cell_id, loader in test_loaders.items():
    print(f"  - Test cell {cell_id}: {len(loader.dataset)} cycles ({len(loader)} batches)")


## 2. Model Imports & Architecture Verification
Importing proposed TE-Q-Transformer and baseline model suite from `models.proposed` and `models.baselines`.


In [ ]:
from models.proposed import TEQTransformer, rich_entangler_config
from models.baselines import (
    CNN1DModel,
    TCNModel,
    DLinearSOHModel,
    TransformerModel,
    PatchTSTSOHModel,
    ITransformerSOHModel,
    QLSTMSOHModel,
    QNNGRUModel,
    LSTMModel,
    GRUModel,
    list_baselines,
)

models_dict = {
    "TE-Q-Transformer (Proposed)": TEQTransformer(rich_entangler_config()),
    "LSTM": LSTMModel(),
    "GRU": GRUModel(),
    "CNN1D": CNN1DModel(),
    "TCN": TCNModel(),
    "DLinear": DLinearSOHModel(),
    "Transformer": TransformerModel(),
    "PatchTST": PatchTSTSOHModel(),
    "iTransformer": ITransformerSOHModel(),
    "QLSTM": QLSTMSOHModel(),
    "QNN_GRU": QNNGRUModel(),
}

print("=" * 70)
print(f"{'Model Name':<30} | {'Trainable Parameters':<20}")
print("=" * 70)
for name, m in models_dict.items():
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name:<30} | {n_params:<20,d}")
print("=" * 70)


## 3. Evaluation Functions & Reference Metrics


In [ ]:
from src.eval.metrics import calculate_metrics, calculate_macro_metrics

def evaluate_model_on_nasa(model: nn.Module, test_loaders: dict, device: torch.device) -> dict:
    model.eval()
    model.to(device)
    per_cell = {}
    with torch.no_grad():
        for cell_id, loader in test_loaders.items():
            preds, actuals = [], []
            for bx, by in loader:
                bx = bx.to(device)
                out = model(bx)
                preds.append(out.cpu().numpy())
                actuals.append(by.numpy())
            y_pred = np.concatenate(preds)
            y_true = np.concatenate(actuals)
            per_cell[cell_id] = {
                "metrics": calculate_metrics(y_true, y_pred),
                "preds": y_pred,
                "actuals": y_true,
            }
    macro = calculate_macro_metrics([res["metrics"] for res in per_cell.values()])
    return {"per_cell": per_cell, "macro": macro}


## 4. Evaluate Pre-Trained TE-Q-Transformer Checkpoint
Loading pre-trained weights from `artifacts/nasa/01_rich_entangler/nasa_teq_transformer_best.pth`.


In [ ]:
teq_model = models_dict["TE-Q-Transformer (Proposed)"]
ckpt_path = REPO_ROOT / "artifacts" / "nasa" / "01_rich_entangler" / "nasa_teq_transformer_best.pth"

if ckpt_path.exists():
    state_dict = torch.load(ckpt_path, map_location="cpu")
    teq_model.load_state_dict(state_dict)
    print(f"[Evaluation] Successfully loaded checkpoint from {ckpt_path}")
    teq_results = evaluate_model_on_nasa(teq_model, test_loaders, DEVICE)
    print("\n[TE-Q-Transformer NASA Held-Out Results]:")
    print(f"  Macro RMSE: {teq_results['macro']['RMSE']:.5f}")
    print(f"  Macro MAE:  {teq_results['macro']['MAE']:.5f}")
    print(f"  Macro R2:   {teq_results['macro']['R2']:.5f}")
    print("\nPer-Cell Breakdown:")
    for cell_id, res in teq_results['per_cell'].items():
        m = res['metrics']
        print(f"  - {cell_id:<12}: RMSE={m['RMSE']:.5f} | MAE={m['MAE']:.5f} | R2={m['R2']:.5f}")
else:
    print(f"[Warning] Checkpoint not found at {ckpt_path}. Training from scratch or checking results tables.")


## 5. Audited Benchmark Comparison Table
Displaying the complete locked 11-model comparison table from `results/tables/all_models_benchmark_metrics.csv` and `results/tables/proposed_model_reference.csv`.


In [ ]:
bench_csv = RESULT_ROOT / "tables" / "all_models_benchmark_metrics.csv"
ref_csv = RESULT_ROOT / "tables" / "proposed_model_reference.csv"

if bench_csv.exists() and ref_csv.exists():
    df_baselines = pd.read_csv(bench_csv)
    df_prop = pd.read_csv(ref_csv)
    
    # Format table for clean display
    cols = ["model", "family", "rmse", "mae", "r2", "trainable_params"]
    df_b = df_baselines[cols].copy()
    df_p = pd.DataFrame([{
        "model": "TE-Q-Transformer (Proposed)",
        "family": "Quantum-Physics-Transformer",
        "rmse": float(df_prop["rmse"].iloc[0]),
        "mae": float(df_prop["mae"].iloc[0]),
        "r2": float(df_prop["r2"].iloc[0]),
        "trainable_params": int(df_prop["trainable_params"].iloc[0]),
    }])
    
    full_table = pd.concat([df_p, df_b], ignore_index=True)
    full_table = full_table.sort_values(by="rmse").reset_index(drop=True)
    display(full_table)
else:
    print("Benchmark tables not found in results/tables/. Please ensure results are consolidated.")


## 6. SOH Trajectory Tracking Visualization


In [ ]:
if ckpt_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (cell_id, res) in zip(axes, teq_results['per_cell'].items()):
        y_true = res['actuals']
        y_pred = res['preds']
        cycles = np.arange(len(y_true))
        ax.plot(cycles, y_true, 'k-', lw=2, label="Actual SOH")
        ax.plot(cycles, y_pred, 'r--', lw=2, label="TE-Q-Transformer")
        ax.set_title(f"Cell {cell_id} (RMSE: {res['metrics']['RMSE']:.4f}, R²: {res['metrics']['R2']:.3f})")
        ax.set_xlabel("Discharge Cycle")
        ax.set_ylabel("State of Health (SOH)")
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()
